# Frozen probe-control validation

## tl;dr

This notebook validates the frozen C0/C1 transform controls and C4 degradation control before showing reader-facing evidence. The executed summary below reports exact observed values from the saved artifacts, separating structural gates, observations, and caveats; it does not hard-code scientific findings or pass thresholds.

The default Great Lakes paths point to the artifact checkout at `/home/jiamingp/diffusion_models_repo` and the pinned source checkout at `/scratch/huterer_root/huterer0/jiamingp/probe_controls_code_456f01a`. Set `PROBE_CONTROLS_RESULTS_ROOT` or `PROBE_CONTROLS_CODE_ROOT` visibly when your layout differs. The expected source check is equivalent to `git rev-parse HEAD`; the artifact manifest is `local/nf_conditional_bias_probe/manifest.json`.

## Context & Methods

C0 tests symmetry and translation stability of a frozen VGG cosmology probe. C1 orders low-pass and high-pass scale cuts across sharp and Hann windows, with an FFT round-trip null. C4 applies measured-transfer and Gaussian-smoothing controls to real maps and compares them with generated maps.

### Key Assumptions

- The source checkout is frozen at the exact expected commit and the output manifests are clean.
- Held-out simulations are exactly 900 through 931, and each analytical grain is unique.
- CSV validation is chunked; the large prediction files are not loaded fully by default.
- C4 matching of two-point power does not match the one-point PDF or higher-order structure.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

class MissingInputError(FileNotFoundError):
    pass

PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', '/home/jiamingp/diffusion_models_repo')).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get('PROBE_CONTROLS_RESULTS_ROOT', str(PROJECT_DIR / 'results' / 'nf_conditional_bias_probe'))).expanduser().resolve()
CODE_ROOT = Path(os.environ.get('PROBE_CONTROLS_CODE_ROOT', '/scratch/huterer_root/huterer0/jiamingp/probe_controls_code_456f01a')).expanduser().resolve()
EXPECTED_COMMIT = 'dced4f8928efe248d819a72560ef61a099d0c4a3'
EXPECTED_RUNS = {'nf_cond_bias_hi_u128_d2p07_n128_200k', 'nf_cond_bias_hi_u128_d2p14_n16384_200k'}
EXPECTED_HELDOUT = np.arange(900, 932, dtype=np.int64)
PARAMETER_ORDER = ('Omega_m', 'sigma_8', 'A_SN1', 'A_AGN1', 'A_SN2', 'A_AGN2')
PARAMETERS = set(PARAMETER_ORDER)
FOCUS_PARAMETER = os.environ.get('PROBE_CONTROLS_PARAMETER', 'Omega_m')
if FOCUS_PARAMETER not in PARAMETERS:
    raise ValueError(f'PROBE_CONTROLS_PARAMETER must be one of {sorted(PARAMETERS)}; got {FOCUS_PARAMETER!r}')
C4_LIMITATION = 'Matching two-point power does not match the one-point PDF or higher-order structure.'

TRANSFORM_DIR = RESULTS_ROOT / 'transform_controls'
C4_DIR = RESULTS_ROOT / 'degradation_controls'
MANIFEST_PATH = PROJECT_DIR / 'local' / 'nf_conditional_bias_probe' / 'manifest.json'
ENCODER_PATH = RESULTS_ROOT / 'encoder' / 'vgg_mlp_encoder.npz'
HEAD_PATH = RESULTS_ROOT / 'encoder' / 'vgg_mlp_encoder.pkl'
VGG_WEIGHTS_PATH = Path(os.environ.get('PROBE_CONTROLS_VGG_WEIGHTS', '/scratch/huterer_root/huterer0/jiamingp/torch_cache/hub/checkpoints/vgg16-397923af.pth')).expanduser().resolve()
TRANSFORM_CSV = TRANSFORM_DIR / 'probe_transform_predictions.csv'
C4_CSV = C4_DIR / 'probe_degradation_predictions.csv'

print('PROJECT_DIR =', PROJECT_DIR)
print('RESULTS_ROOT =', RESULTS_ROOT)
print('CODE_ROOT =', CODE_ROOT)
print('EXPECTED_COMMIT =', EXPECTED_COMMIT)

## Data

The gates below validate provenance and schemas before plotting. Missing inputs fail clearly rather than producing an empty or misleading report. Set `PROBE_CONTROLS_PARAMETER` to focus the C4 small multiples on another validated parameter.

On Great Lakes, execute with a job-local output directory (Jupyter must be available in the selected environment):

```bash
/home/jiamingp/venvs/cosmodiff_nf_class/bin/python -m jupyter nbconvert --execute --to notebook \
  --output-dir /home/jiamingp/diffusion_models_repo/notebooks/executed \
  --output nf_probe_control_validation.executed.ipynb \
  /home/jiamingp/diffusion_models_repo/notebooks/nf_probe_control_validation.ipynb
```

In [ ]:
def require_file(path):
    path = Path(path)
    if not path.is_file():
        raise MissingInputError(f'Missing required input: {path}')
    return path

def git_head(root):
    metadata = Path(root) / '.git'
    if metadata.is_file():
        pointer = metadata.read_text().strip().split(':', 1)[-1].strip()
        metadata = (metadata.parent / pointer).resolve()
    head = require_file(metadata / 'HEAD').read_text().strip()
    if head.startswith('ref:'):
        return require_file(metadata / head.split(':', 1)[1].strip()).read_text().strip()
    return head

if git_head(CODE_ROOT) != EXPECTED_COMMIT:
    raise RuntimeError(f'Expected CODE_ROOT commit {EXPECTED_COMMIT}; found {git_head(CODE_ROOT)}')

required_inputs = [
    ENCODER_PATH, HEAD_PATH, MANIFEST_PATH, VGG_WEIGHTS_PATH,
    TRANSFORM_CSV, C4_CSV,
    TRANSFORM_DIR / 'probe_transform_metrics.json',
    TRANSFORM_DIR / 'c0_symmetry_summary.json',
    TRANSFORM_DIR / 'c1_scale_cut_summary.json',
    TRANSFORM_DIR / 'manifest.json',
    C4_DIR / 'probe_degradation_metrics.json',
    C4_DIR / 'power_transfer_curves.json',
    C4_DIR / 'field_histograms.json',
    C4_DIR / 'manifest.json',
]
for input_path in required_inputs:
    require_file(input_path)

def strict_json(path):
    def reject_constant(value):
        raise ValueError(f'Non-strict JSON constant {value} in {path}')
    def reject_duplicate_keys(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f'Duplicate JSON key {key!r} in {path}')
            result[key] = value
        return result
    return json.loads(Path(path).read_text(), parse_constant=reject_constant, object_pairs_hook=reject_duplicate_keys)

def assert_clean_manifest(payload, path):
    if not isinstance(payload, dict) or not isinstance(payload.get('git'), dict):
        raise ValueError(f'Manifest lacks Git provenance: {path}')
    git = payload['git']
    if git.get('revision') != EXPECTED_COMMIT or git.get('dirty') is not False:
        raise ValueError(f'Manifest is not clean at the expected commit: {path}')

transform_manifest = strict_json(TRANSFORM_DIR / 'manifest.json')
c4_manifest = strict_json(C4_DIR / 'manifest.json')
assert_clean_manifest(transform_manifest, TRANSFORM_DIR / 'manifest.json')
assert_clean_manifest(c4_manifest, C4_DIR / 'manifest.json')

with np.load(ENCODER_PATH, allow_pickle=True) as encoder_data:
    heldout = np.asarray(encoder_data['heldout_indices'], dtype=np.int64)
if not np.array_equal(heldout, EXPECTED_HELDOUT):
    raise ValueError(f'heldout_indices must be 900..931; found {heldout.tolist()}')

manifest_rows = strict_json(MANIFEST_PATH)
if not isinstance(manifest_rows, list):
    raise ValueError('The generated-run manifest must be a JSON list')
manifest_runs = {str(row.get('run_name')) for row in manifest_rows}
if not EXPECTED_RUNS.issubset(manifest_runs):
    raise ValueError(f'Manifest is missing expected runs: {sorted(EXPECTED_RUNS - manifest_runs)}')
print('Structural input gates passed for', len(manifest_rows), 'manifest rows')

In [ ]:
TRANSFORM_REQUIRED = {
    'transform', 'transform_family', 'k_cut', 'k_cut_over_knyq', 'window',
    'dihedral_g', 'roll_dx', 'roll_dy', 'sim_index', 'z_index',
    'parameter', 'theta_true', 'theta_pred', 'out_of_range_fraction',
}
C4_REQUIRED = TRANSFORM_REQUIRED | {'source', 'run_name', 'dataset_size'}
EXPECTED_TRANSFORM_LINES = 1_990_657
EXPECTED_C4_LINES = 73_729

def validate_prediction_csv(path, expected_lines, required_columns, key_columns, require_all_simulations=True):
    header = pd.read_csv(path, nrows=0)
    missing = sorted(set(required_columns) - set(header.columns))
    if missing:
        raise ValueError(f'{path} is missing required columns: {missing}')
    with Path(path).open('rb') as stream:
        line_count = sum(1 for _ in stream)
    if line_count != expected_lines:
        raise ValueError(f'{path} has {line_count} lines; expected {expected_lines}')
    seen_hashes = set()
    transforms, transform_families, parameters, simulations, run_names = set(), set(), set(), set(), set()
    rows = 0
    usecols = sorted(set(required_columns) | set(key_columns))
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=200_000):
        rows += len(chunk)
        transforms.update(chunk['transform'].dropna().astype(str))
        transform_families.update(chunk['transform_family'].dropna().astype(str))
        parameters.update(chunk['parameter'].dropna().astype(str))
        simulations.update(chunk['sim_index'].dropna().astype(int))
        if 'run_name' in chunk:
            run_names.update(chunk['run_name'].dropna().astype(str))
        keys = chunk[key_columns].astype(object).where(chunk[key_columns].notna(), '<NA>')
        hashes = pd.util.hash_pandas_object(keys, index=False).astype('uint64')
        if hashes.duplicated().any() or set(hashes.tolist()) & seen_hashes:
            raise ValueError(f'{path} contains duplicate analytical grains')
        seen_hashes.update(hashes.tolist())
    if rows + 1 != line_count:
        raise ValueError(f'{path} row/line reconciliation failed')
    if (require_all_simulations and simulations != set(EXPECTED_HELDOUT)) or (not require_all_simulations and not simulations.issubset(set(EXPECTED_HELDOUT))) or parameters != PARAMETERS:
        raise ValueError(f'{path} does not cover heldout simulations and all six parameters')
    return {'transforms': transforms, 'transform_families': transform_families, 'parameters': parameters, 'simulations': simulations, 'run_names': run_names}

transform_coverage = validate_prediction_csv(
    TRANSFORM_CSV, EXPECTED_TRANSFORM_LINES, TRANSFORM_REQUIRED,
    ['transform', 'transform_family', 'sim_index', 'z_index', 'parameter'],
)
c4_coverage = validate_prediction_csv(
    C4_CSV, EXPECTED_C4_LINES, C4_REQUIRED,
    ['transform', 'transform_family', 'source', 'run_name', 'dataset_size', 'sim_index', 'z_index', 'parameter'],
    require_all_simulations=False,
)
transform_names = transform_coverage['transforms']
transform_families = transform_coverage['transform_families']
c4_runs = c4_coverage['run_names']
if 'identity' not in transform_names:
    raise ValueError('C0 identity coverage is incomplete')
if not {'lowpass', 'highpass', 'fft_roundtrip_null'}.issubset(transform_families):
    raise ValueError(f'C1 transform-family coverage is incomplete: {sorted(transform_families)}')
if not EXPECTED_RUNS.issubset(c4_runs):
    raise ValueError('C4 run coverage is incomplete')
print('CSV gates passed without loading either large prediction file fully')

## Results

The following cells use the validated summary JSONs for compact plots. Error bars are the stored uncertainty intervals; no arbitrary scientific pass threshold is introduced.

### C0 — symmetry and translation stability

The family summaries compare dihedral and roll spreads with the within-simulation identity baseline. The worst cases panel is descriptive and should not be read as a binary acceptance test.

In [ ]:
c0_summary = strict_json(TRANSFORM_DIR / 'c0_symmetry_summary.json')
family_summary = pd.DataFrame.from_dict(c0_summary['family_summary'], orient='index').reset_index(names='family')
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
x = np.arange(len(family_summary))
for axis, value, low, high, title in [(axes[0], 'median_std_ratio', 'median_std_ratio_ci_low', 'median_std_ratio_ci_high', 'C0 transform spread / identity spread'), (axes[1], 'median_range_ratio', 'median_range_ratio_ci_low', 'median_range_ratio_ci_high', 'C0 transform range / identity range')]:
    y = family_summary[value].to_numpy(float)
    lower = y - family_summary[low].to_numpy(float)
    upper = family_summary[high].to_numpy(float) - y
    axis.errorbar(x, y, yerr=[lower, upper], fmt='o', color='tab:blue', capsize=4)
    axis.set_xticks(x, family_summary['family'])
    axis.set_title(title)
    axis.set_ylabel('ratio (descriptive)')
    axis.grid(alpha=0.25)
fig.suptitle('C0 symmetry and translation stability')
plt.show()

worst = pd.DataFrame(c0_summary['per_slice']).sort_values('std_over_within_sim_std', ascending=False).head(10)
display(worst[['family', 'sim_index', 'z_index', 'std_over_within_sim_std', 'range_over_within_sim_range']])

### C1 — ordered scale-cut curves

Curves are shown in increasing k-cut order for low-pass and high-pass arms. Line style distinguishes sharp/Hann windows and markers distinguish transform families; the FFT round-trip null is a neutral baseline.

In [ ]:
c1_summary = strict_json(TRANSFORM_DIR / 'c1_scale_cut_summary.json')
c1 = pd.DataFrame(c1_summary['curves'])
c1 = c1[c1['grain'] == 'per_cosmology'].copy()
def assert_unique_grain(frame, columns, label):
    duplicate = frame.duplicated(columns, keep=False)
    if duplicate.any():
        sample = frame.loc[duplicate, columns].head(3).to_dict('records')
        raise ValueError(f'{label} has duplicate analytical grains: {sample}')

assert_unique_grain(c1, ['parameter', 'transform', 'transform_family', 'window', 'k_cut'], 'C1 summary')
parameter_facets = list(PARAMETER_ORDER)
if set(parameter_facets) != set(c1['parameter'].astype(str)):
    raise ValueError(f'C1 parameter coverage is incomplete: {parameter_facets}')
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False, constrained_layout=True)
styles = {'sharp': '-', 'hann': '--'}
markers = {'lowpass': 'o', 'highpass': 's'}
for axis, parameter in zip(axes.ravel(), parameter_facets):
    subset = c1[c1['parameter'].astype(str) == parameter]
    for (family, window), group in subset[subset['transform_family'].isin(['lowpass', 'highpass'])].groupby(['transform_family', 'window']):
        group = group.sort_values('k_cut')
        y = group['rmse'].to_numpy(float)
        yerr = [y - group['rmse_ci_low'].to_numpy(float), group['rmse_ci_high'].to_numpy(float) - y]
        axis.errorbar(group['k_cut'], y, yerr=yerr, color='tab:blue' if family == 'lowpass' else 'tab:orange', linestyle=styles[window], marker=markers[family], capsize=2, label=f'{family}, {window}')
    identity = subset[subset['transform'] == 'identity']
    null = subset[subset['transform_family'] == 'fft_roundtrip_null']
    if len(identity) != 1 or len(null) != 1:
        raise ValueError(f'C1 parameter-specific identity/null reference missing for {parameter}')
    axis.axhline(float(identity['rmse'].iloc[0]), color='0.25', linestyle='-.', label='identity')
    axis.axhline(float(null['rmse'].iloc[0]), color='0.45', linestyle=':', label='FFT round-trip null')
    axis.set_title(parameter)
    axis.set_xlabel('k cut (ordered)')
    axis.set_ylabel('RMSE (descriptive)')
    axis.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
fig.suptitle('C1 ordered k-cut probe RMSE by parameter; lines show stored uncertainty')
plt.show()
compact_c1 = c1[['parameter', 'transform_family', 'window', 'k_cut', 'rmse', 'rmse_ci_low', 'rmse_ci_high', 'bias', 'bias_ci_low', 'bias_ci_high', 'slope', 'slope_ci_low', 'slope_ci_high', 'out_of_range_fraction']]
display(compact_c1.sort_values(['parameter', 'transform_family', 'window', 'k_cut']).head(24))

### C4 — grouped probe metrics, transfer, and one-point views

C4 groups original-real, measured-transfer, Gaussian, and generated metrics by run. Power ratio/transfer and field histograms are shown as supporting views. The limitation remains explicit: matching two-point power does not match the one-point PDF or higher-order structure.

In [ ]:
c4_metrics = strict_json(C4_DIR / 'probe_degradation_metrics.json')
power = strict_json(C4_DIR / 'power_transfer_curves.json')
histograms = strict_json(C4_DIR / 'field_histograms.json')
if c4_metrics.get('limitation') != C4_LIMITATION or power.get('limitation') != C4_LIMITATION:
    raise ValueError('C4 limitation text is missing or changed')
metrics = pd.DataFrame(c4_metrics['metrics'])
metrics = metrics[metrics['grain'] == 'per_cosmology'].copy()
assert_unique_grain(metrics, ['parameter', 'grain', 'source', 'run_name', 'transform'], 'C4 metrics')
source_order = ['real_original', 'real_measured_transfer', 'real_gaussian', 'generated']
if set(metrics['source'].astype(str)) != set(source_order):
    raise ValueError('C4 source coverage is incomplete')
baseline = metrics[metrics['source'] == 'real_original'].copy()
if len(baseline) == 0 or set(baseline['parameter']) != PARAMETERS:
    raise ValueError('C4 real_original baseline is incomplete')
baseline_rows = []
for run_name in sorted(EXPECTED_RUNS):
    repeated = baseline.copy()
    repeated['run_name'] = run_name
    repeated['transform'] = 'identity__baseline__' + run_name
    baseline_rows.append(repeated)
comparison = pd.concat([metrics[metrics['source'] != 'real_original']] + baseline_rows, ignore_index=True)
assert_unique_grain(comparison, ['parameter', 'grain', 'source', 'run_name', 'transform'], 'C4 comparison')
if set(comparison['run_name'].astype(str)) != EXPECTED_RUNS:
    raise ValueError('C4 comparison does not cover both runs')
metrics_to_plot = [('rmse', 'RMSE'), ('bias', 'Bias'), ('slope', 'Slope')]
source_short_labels = {'real_original': 'original real', 'real_measured_transfer': 'measured transfer', 'real_gaussian': 'Gaussian', 'generated': 'generated'}
source_styles = {'real_original': ('0.25', 'o', '-'), 'real_measured_transfer': ('tab:blue', 's', '--'), 'real_gaussian': ('tab:orange', '^', ':'), 'generated': ('0.10', 'D', '-.')}
run_names = sorted(EXPECTED_RUNS)
if set(run_names) != set(comparison['run_name'].astype(str)):
    raise ValueError('C4 comparison run set is not exactly the expected run set')
RUN_LABELS = {'nf_cond_bias_hi_u128_d2p07_n128_200k': 'N=128, d=2.07', 'nf_cond_bias_hi_u128_d2p14_n16384_200k': 'N=16,384, d=2.14'}
if set(RUN_LABELS) != set(run_names):
    raise ValueError('C4 run labels do not cover the expected runs')
fig, axes = plt.subplots(2, 3, figsize=(17, 9), squeeze=False, constrained_layout=True)
for row_index, run_name in enumerate(run_names):
    run_label = RUN_LABELS[run_name]
    run_rows = comparison[(comparison['run_name'] == run_name) & (comparison['parameter'] == FOCUS_PARAMETER)]
    if set(run_rows['source']) != set(source_order):
        raise ValueError(f'C4 source comparison is incomplete for {run_name}')
    run_rows = run_rows.set_index('source').loc[source_order].reset_index()
    x = np.arange(len(source_order))
    for column_index, (metric, label) in enumerate(metrics_to_plot):
        axis = axes[row_index, column_index]
        y = run_rows[metric].to_numpy(float)
        lower = y - run_rows[f'{metric}_ci_low'].to_numpy(float)
        upper = run_rows[f'{metric}_ci_high'].to_numpy(float) - y
        for index, row in run_rows.iterrows():
            color, marker, _ = source_styles[row['source']]
            axis.errorbar(x[index], y[index], yerr=[[lower[index]], [upper[index]]], color=color, marker=marker, linestyle='none', capsize=3, label=source_short_labels[row['source']])
        if metric == 'bias':
            axis.axhline(0, color='0.45', linestyle=':', linewidth=1, label='zero reference')
        elif metric == 'slope':
            axis.axhline(1, color='0.45', linestyle=':', linewidth=1, label='unit reference')
        axis.set_xticks(x, [source_short_labels[source] for source in source_order], rotation=28, ha='right')
        axis.set_title(f'Run {row_index + 1}: {run_label} — {label}')
        axis.set_ylabel(label + ' (descriptive)')
        axis.grid(axis='y', alpha=0.25)
        if row_index == 0 and column_index == 2:
            axis.legend(fontsize=8, loc='best')
fig.suptitle(f'C4 grouped probe metrics for {FOCUS_PARAMETER}; stored CIs')
plt.show()
display(comparison[comparison['parameter'] == FOCUS_PARAMETER][['run_name', 'source', 'rmse', 'rmse_ci_low', 'rmse_ci_high', 'bias', 'bias_ci_low', 'bias_ci_high', 'slope', 'slope_ci_low', 'slope_ci_high']].sort_values(['run_name', 'source']))

RUN_COLORS = {run_names[0]: 'tab:blue', run_names[1]: 'tab:orange'}
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
for run_name in run_names:
    run = power['runs'][run_name]
    color = RUN_COLORS[run_name]
    axes[0].plot(run['k_bins'], run['power_ratio'], color=color, marker='o', linestyle='-', label=f'{run_name}: generated / real')
    axes[1].plot(run['k_bins'], run['measured_transfer'], color=color, marker='s', linestyle='--', label=f'{run_name}: measured transfer')
    axes[1].plot(run['k_bins'], run['gaussian_transfer'], color=color, marker='^', linestyle=':', label=f'{run_name}: Gaussian transfer')
axes[0].set_title('C4 generated / real power ratio')
axes[0].set_xlabel('k bin'); axes[0].set_ylabel('power ratio')
axes[1].set_title('C4 measured and Gaussian transfer')
axes[1].set_xlabel('k bin'); axes[1].set_ylabel('transfer amplitude')
for axis in axes: axis.legend(fontsize=8); axis.grid(alpha=0.25)
plt.show()

fig, axes = plt.subplots(1, len(EXPECTED_RUNS), figsize=(15, 4), squeeze=False, constrained_layout=True)
hist_baseline = histograms['real_original']
for axis, run_name in zip(axes.ravel(), sorted(EXPECTED_RUNS)):
    run = histograms['runs'][run_name]
    series = [('real_original', hist_baseline), ('real_measured_transfer', run['real_measured_transfer']), ('real_gaussian', run['real_gaussian']), ('generated', run['generated'])]
    for source, values in series:
        color, _, linestyle = source_styles[source]
        edges = np.asarray(values['bin_edges'], dtype=float)
        heights = np.asarray(values['hist'], dtype=float)
        if len(edges) != len(heights) + 1:
            raise ValueError(f'Histogram bin schema is invalid for {run_name}: {source}')
        centers = 0.5 * (edges[:-1] + edges[1:])
        axis.hist(centers, bins=edges, weights=heights, histtype='step', color=color, linestyle=linestyle, linewidth=1.5, label=source_short_labels[source])
    axis.set_title(f'C4 one-point field PDF: {run_name}')
    axis.set_xlabel('field value'); axis.set_ylabel('density')
    axis.legend(fontsize=7); axis.grid(alpha=0.2)
plt.show()

## Takeaways

The notebook status is intentionally limited to evidence quality, not a scientific threshold. `Ready to share` means structural and provenance gates plus an executed review; `Share with caveats` is appropriate when the gates pass but the controls remain diagnostic; `Needs revision` is reserved for failed gates or unresolved schema/provenance issues. The exact C0 ratios and the closest non-generated C4 source are computed from the validated artifacts below.

C4 limitation: matching two-point power does not match the one-point PDF or higher-order structure. Open questions and any observed asymmetries should be reported from the validated tables above rather than pre-written here.

In [ ]:
STATUS_LABELS = ('Ready to share', 'Share with caveats', 'Needs revision')
structural_status = 'verified'
status = 'Share with caveats' if structural_status == 'verified' else 'Needs revision'
c0_observed = family_summary[['family', 'median_std_ratio', 'median_std_ratio_ci_low', 'median_std_ratio_ci_high', 'median_range_ratio', 'median_range_ratio_ci_low', 'median_range_ratio_ci_high']].copy()
closest_rows = []
for run_name in sorted(EXPECTED_RUNS):
    run_rows = comparison[(comparison['run_name'] == run_name) & (comparison['parameter'] == FOCUS_PARAMETER)].copy()
    generated_value = float(run_rows.loc[run_rows['source'] == 'generated', 'rmse'].iloc[0])
    alternatives = run_rows[run_rows['source'] != 'generated'].copy()
    alternatives['absolute_rmse_difference_from_generated'] = (alternatives['rmse'] - generated_value).abs()
    closest = alternatives.sort_values(['absolute_rmse_difference_from_generated', 'source']).iloc[0]
    closest_rows.append({'run_name': run_name, 'closest_non_generated_source_by_rmse': closest['source'], 'generated_rmse': generated_value, 'closest_rmse': float(closest['rmse']), 'absolute_rmse_difference_from_generated': float(closest['absolute_rmse_difference_from_generated'])})
display(Markdown(f'## tl;dr\n\n**Status: {status}** — structural status: **{structural_status}**. The observations below are exact values from the saved summaries; no scientific pass threshold is applied. C4 remains subject to the limitation that matching two-point power does not match the one-point PDF or higher-order structure.'))
display(Markdown('**Observed C0 family ratios**'))
display(c0_observed)
display(Markdown(f'**Observed C4 closest non-generated source by {FOCUS_PARAMETER} RMSE**'))
display(pd.DataFrame(closest_rows))